# 0. Basic Project Information, Introduction

## ENGI-9874-001 GR_37
### Project Members
```Last Name, First Name```
- Fang, Zhou
- Patel, Vivek Thakorbhai
- Patel, Kevin Harshadkumar

#### Project Name: Library Management System
- A simple system used to practice software design/specification and design patterns
- Patterns used (8)
    - decorator
    - strategy
    - command
    - observer
    - singleton
    - factory (factory method)
    - abstract factory
    - adapter

### How to use the system

Step 1

- Install required packages
    - tkinter (tk)
    - abc (is already a built-in std package)

Step 2

- Run the notebook and the program will show up
    - Add a book with any name of book/author
        - ```factory pattern``` used to create books
        - ```observer pattern``` used for notifying members about new books, the information will be displayed at console to simulate the notification.
    - Add third-party books
        - ```adapter pattern``` used to adapt hard-coded third-party books into the library
        - ```observer pattern``` used for notifying members about new third-party books, the information will be displayed at console to simulate the notification.
    - Add a member 
        - ```abstract factory pattern``` used to create different types of members (Student/Faculty)
        - ```observer pattern``` used to attach new members as observers
    - Search for the book you added with the names
        - ```strategy pattern``` used to implement different search strategies (by title or author)
    - Borrow a book with the given name
        - ```command pattern``` used to handle the borrowing process
        - Add accessories when borrowing a book
            - ```decorator pattern``` used to add accessories to the book
    - Return the book with the given name
        - ```command pattern``` used to handle the return process
    - Show the available books
        - ```command pattern``` used to display all books available for borrowing
    - Show the borrowed books
        - ```command pattern``` used to display all borrowed books
    - Show current registered users
        - ```command pattern``` displays all users

Note: The following classes in the program are implemented using the 
- ```Singleton pattern``` to ensure only one instance exists:
    - `Library`: Manages the collection of books and members.
    - `LibraryNotifier`: Handles the notification system for members.
    - `MemberFactory` and its subclasses (`StudentMemberFactory` and `FacultyMemberFactory`): Ensure there is only one instance of each factory.

# 1. Dependency Installation

In [15]:
%pip install tk

Note: you may need to restart the kernel to use updated packages.


# 2. Program Code / Runnables

## Decorator Pattern

In [16]:
from abc import ABC

# Decorator Pattern
class Book(ABC):
    def __init__(self, title: str, author: str):
        self.title = title
        self.author = author

    def get_description(self) -> str:
        return f"{self.title} by {self.author if self.author.strip() else 'Unkown'}"

class BookDecorator(ABC):

    def __init__(self, title: str, author: str, book: Book):
        self._decoratedBook = book

class BookAccessoryDecorator(BookDecorator):
    def __init__(self, name: str, book: Book):
        # Call the parent class constructor correctly
        super().__init__("", "", book)
        self._name = name

    def get_description(self) -> str:
        return self._decoratedBook.get_description() + f"\nAccessory: {self._name}"

class EBook(Book):
    pass

class PrintBook(Book):
    pass

In [17]:
# test decorator pattern
a = BookAccessoryDecorator("pen", Book("book1", "author1"))
b = BookAccessoryDecorator("pencil", a)
c = BookAccessoryDecorator("erasor", b)
print(c.get_description())

book1 by author1
Accessory: pen
Accessory: pencil
Accessory: erasor


## Factory Method Pattern

In [18]:
# Factory Method Pattern
class BookFactory:
    @staticmethod
    def create_book(book_type: str, title: str, author: str) -> 'Book':
        if book_type == "EBook":
            return EBook(title, author)
        elif book_type == "PrintBook":
            return PrintBook(title, author)
        else:
            return None

In [19]:
from abc import ABC, abstractmethod
class Observer(ABC):
    @abstractmethod
    def update(self, *args, **kwargs) -> None:
        pass

class Member(Observer):
    
    @abstractmethod
    def __init__(self):
        pass

class StudentMember(Member):
    def __init__(self, name: str):
        self.name = name

    def update(self, *args, **kwargs) -> None:
        book = kwargs.get('book')
        if book:
            print(f"Student {self.name} notified of new book: {book.get_description()}")

class FacultyMember(Member):
    def __init__(self, name: str):
        self.name = name

    def update(self, *args, **kwargs) -> None:
        book = kwargs.get('book')
        if book:
            print(f"Faculty {self.name} notified of new book: {book.get_description()}")

## Abstract Factory Pattern and Singleton Pattern

In [20]:
from abc import ABC, abstractmethod
# Abstract Factory Pattern and Singleton Pattern for MemberFactory
class MemberFactory(ABC):
    _instances: dict['MemberFactory', 'MemberFactory'] = {}

    def __new__(cls, *args, **kwargs) -> 'MemberFactory':
        if cls not in cls._instances:
            instance = super().__new__(cls)
            cls._instances[cls] = instance
        return cls._instances[cls]

    @abstractmethod
    def create_member(self, name: str) -> 'Member':
        pass

class StudentMemberFactory(MemberFactory):
    def create_member(self, name: str) -> 'StudentMember':
        return StudentMember(name)

class FacultyMemberFactory(MemberFactory):
    def create_member(self, name: str) -> 'FacultyMember':
        return FacultyMember(name)

## Observer Pattern

In [21]:
# Observer Pattern
from abc import ABC

class Notifier(ABC):
    _instance: 'Notifier' = None

    def __new__(cls, *args, **kwargs) -> 'Notifier':
        if cls._instance is None:
            cls._instance = super(Notifier, cls).__new__(cls)
        return cls._instance

    def __init__(self):
        if not hasattr(self, '_initialized'):  # Ensure __init__ only runs once to protect _observers
            self._observers: list[Observer] = []
            self._initialized = True

    def attach(self, observer: 'Observer') -> None:
        if observer not in self._observers:
            self._observers.append(observer)

    def detach(self, observer: 'Observer') -> None:
        if observer in self._observers:
            self._observers.remove(observer)

    @abstractmethod
    def notify(self, *args, **kwargs) -> None:
        pass

class LibraryNotifier(Notifier):
    def __init__(self):
        super().__init__()  # Initialize the base Notifier class

    def notify(self, *args, **kwargs) -> None:
        """
        Notify all attached observers with the details of the new book.
        The book is passed as a keyword argument to each observer's update method.
        """
        for observer in self._observers:
            observer.update(self, *args, **kwargs)

## Singleton Pattern

In [22]:
# Singleton Pattern for Library
class Library:
    _instance: 'Library' = None

    # type hints
    books: list['Book']
    borrowed_books: list['Book']
    members: list['Member']

    def __new__(cls) -> 'Library':
        if cls._instance is None:
            cls._instance = super(Library, cls).__new__(cls)
            cls._instance.books= []
            cls._instance.borrowed_books = []
            cls._instance.members = []
        return cls._instance

    def borrow_book(self, book: 'Book') -> bool:
        if book in self.books:
            self.books.remove(book)
            self.borrowed_books.append(book)
            return True
        return False

    def return_book(self, book: 'Book') -> bool:
        if book in self.borrowed_books:
            self.borrowed_books.remove(book)
            self.books.append(book)
            return True
        return False

## Strategy Pattern

In [23]:
from typing import Tuple

# Strategy Pattern
class SearchStrategy(ABC):
    @abstractmethod
    def search(self, library: Library, query: str) -> Tuple[list[Book], list[Book]]:
        """
        Returns:
        - a tulple consists of two lists: a list of available Books, and a list borrowed Books.
        """
        pass

class TitleSearchStrategy(SearchStrategy):

    def search(self, library: Library, query: str) -> Tuple[list[Book], list[Book]]:
        available_books = [book for book in library.books if query.lower() in book.title.lower()]
        borrowed_books = [book for book in library.borrowed_books if query.lower() in book.title.lower()]
        return available_books, borrowed_books

class AuthorSearchStrategy(SearchStrategy):

    def search(self, library: Library, query: str) -> Tuple[list[Book], list[Book]]:
        available_books = [book for book in library.books if query.lower() in book.author.lower()]
        borrowed_books = [book for book in library.borrowed_books if query.lower() in book.author.lower()]
        return available_books, borrowed_books

## Adapter Pattern

In [24]:
# Adapter Pattern
class ThirdPartyBookAPI:
    """As lagecy books that need to be adaptered"""
    def get_books(self):
        return [{"title": "ThirdPartyBook1", "author": "Author1"}, {"title": "ThirdPartyBook2", "author": "Author2"}]
    
class LegacyBook:

    def __init__(self, dict: dict):
        self.bookName = dict['title']
        self.bookAuthor = dict['author']

class BookAdapter(Book):

    def __init__(self, third_party_book):
        self.adaptee = LegacyBook(third_party_book)

    def get_description(self) -> str:
        return f"{self.adaptee.bookName} by {self.adaptee.bookAuthor if self.adaptee.bookAuthor.strip() else 'Unknown'}"

## Command Pattern

In [25]:
from abc import ABC, abstractmethod
from typing import Tuple, Dict, Optional

# Command Pattern
class Command(ABC):
    @abstractmethod
    def execute(self, *args, **kwargs) -> Tuple[bool, Dict[str, Optional[str]]]:
        pass

class BorrowBookWithAccessoriesCommand(Command):

    def __init__(self, *args, **kwargs):
        """
        Initializes the DisplayUsersCommand.

        Keyword Arguments:
        - library ('Library'): The library instance.
        """
        self.library: 'Library' = kwargs.get('library')

    def execute(self, *args, **kwargs) -> Tuple[bool, Dict[str, Optional[str]]]:
        """
        Executes the borrow book with accessories command.

        Parameters:
        *args: Variable length argument list. (Not used in this implementation but provided for flexibility)
        **kwargs: Arbitrary keyword arguments. Expected keys include:
            - title (str): The title of the book to be borrowed.
            - accessories (list): The list of accessories to add to the book.

        Returns:
        Tuple[bool, Dict[str, Optional[str]]]: A tuple where:
            - The first element is a boolean indicating whether the command was successful (True) or not (False).
            - The second element is a dictionary containing additional information:
                - "message" (str): A message indicating the result of the command.
        """
        title: str = kwargs.get('title')
        accessories: list[str] = kwargs.get('accessories', [])

        book: Optional['Book'] = next((book for book in self.library.books if book.title == title), None)
        if not book or not self.library.borrow_book(book):
            return False, {"message": f"Book '{title}' not found or already borrowed"}

        # Add accessories if any
        success, data = self.__addAccessories(book=book, accessories=accessories)

        if success:
            return True, {"message": f"Book borrowed with accessories: {data['book'].get_description()}"}
        else:
            return False, {"message": "Failed to add accessories to the borrowed book"}
        
    def __addAccessories(self, *args, **kwargs) -> Tuple[bool, Dict[str, Optional[str]]]:
        """
        Utilize the decorator pattern to wrap accessory with book.
        
        Parameters:
        **kwargs:
            - book (Book): The Book to be borrowed.
            - accessories (list[str]): List of accessory names.
        """
        book: Book = kwargs.get('book')
        accessories: list[str] = kwargs.get('accessories', [])

        if not book:
            return False, {"message": "No book provided.", "book": None}

        for accessory in accessories:
            book = BookAccessoryDecorator(accessory, book)

        return True, {"message": f"Book borrowed with accessories: {book.get_description()}", "book": book}

class ReturnBookCommand(Command):
    
    def __init__(self, *args, **kwargs):
        """
        Initializes the ReturnBookCommand.

        Keyword Arguments:
        - library ('Library'): The library instance where the book will be returned to.
        """
        self.library: 'Library' = kwargs.get('library')

    def execute(self, *args, **kwargs) -> Tuple[bool, Dict[str, Optional[str]]]:
        """
        Executes the return book command.

        Parameters:
        *args: Variable length argument list. (Not used in this implementation but provided for flexibility)
        **kwargs: Arbitrary keyword arguments. Expected keys include:
            - title (str): The title of the book to be returned.

        Returns:
        Tuple[bool, Dict[str, Optional[str]]]: A tuple where:
            - The first element is a boolean indicating whether the command was successful (True) or not (False).
            - The second element is a dictionary containing additional information:
                - "message" (str): A message indicating the result of the command.
                - "book" (Optional[str]): The `Book` object description that was returned if the command was successful, or `None` if the operation failed.
        """
        title: str = kwargs.get('title')
        book: Optional['Book'] = next((book for book in self.library.borrowed_books if book.title == title), None)
        if book and self.library.return_book(book):
            return True, {"message": f"Returned book: {book.get_description()}", "book": book.get_description()}
        else:
            return False, {"message": f"Book '{title}' not found in borrowed books", "book": None}

class AddBookCommand(Command):

    def __init__(self, *args, **kwargs):
        """
        Initializes the AddBookCommand.

        Keyword Arguments:
        - library ('Library'): The library instance where the new book will be added.
        - notifier ('Notifier'): The notifier instance that will notify observers of the new book.
        """
        self.library: 'Library' = kwargs.get('library')
        self.notifier: 'Notifier' = kwargs.get('notifier')

    def execute(self, *args, **kwargs) -> Tuple[bool, Dict[str, Optional[str]]]:
        """
        Executes the add book command.

        Parameters:
        *args: Variable length argument list. (Not used in this implementation but provided for flexibility)
        **kwargs: Arbitrary keyword arguments. Expected keys include:
            - book_type (str): The type of book to create (e.g., "EBook", "PrintBook").
            - title (str): The title of the book.
            - author (str): The author of the book.

        Returns:
        Tuple[bool, Dict[str, Optional[str]]]: A tuple where:
            - The first element is a boolean indicating whether the command was successful (True) or not (False).
            - The second element is a dictionary containing additional information:
                - "message" (str): A message indicating the result of the command.
                - "book" (Optional[str]): The `Book` object description that was added if the command was successful, or `None` if the operation failed.
        """
        book_type: str = kwargs.get('book_type')
        title: str = kwargs.get('title')
        author: str = kwargs.get('author')

        # Create the book using the factory
        new_book: Optional['Book'] = BookFactory.create_book(book_type, title, author)
        if new_book:
            # Add the book to the library's collection
            self.library.books.append(new_book)
            
            # Notify all members about the new book
            self.notifier.notify(book=new_book)
            
            return True, {"message": f"Added {new_book.get_description()}", "book": new_book.get_description()}
        else:
            return False, {"message": "Failed to add book", "book": None}


class AddMemberCommand(Command):

    def __init__(self, *args, **kwargs):
        """
        Initializes the AddMemberCommand.

        Keyword Arguments:
        - library ('Library'): The library instance where the member will be added.
        - notifier ('Notifier'): The notifier instance that will notify observers of the new member.
        - All other keyword arguments are assumed to be member type factories, where the key is the member type (lowercased) and the value is the corresponding factory instance.
            - eg Student=MemberFactory()
        """
        self.library: 'Library' = kwargs.get('library')
        self.notifier: 'Notifier' = kwargs.get('notifier')  # Singleton instance
        
        # Assume all other kwargs are factories
        self.factories: Dict[str: MemberFactory] = {key: value for key, value in kwargs.items() if key.lower() not in ['library', 'notifier']}

    def execute(self, *args, **kwargs) -> Tuple[bool, Dict[str, Optional[str]]]:
        """
        Executes the add member command.

        Parameters:
        *args: Variable length argument list. (Not used in this implementation but provided for flexibility)
        **kwargs: Arbitrary keyword arguments. Expected keys include:
            - name (str): The name of the member to add.
            - member_type (str): The type of member to add (e.g., "Student", "Faculty").

        Returns:
        Tuple[bool, Dict[str, Optional[str]]]: A tuple where:
            - The first element is a boolean indicating whether the command was successful (True) or not (False).
            - The second element is a dictionary containing additional information:
                - "message" (str): A message indicating the result of the command.
                - "member" (Optional[str]): The member type and name that was added if the command was successful, or `None` if the operation failed.
        """
        name: str = kwargs.get('name')
        member_type: str = kwargs.get('member_type')

        # Attempt to find the appropriate factory using the member_type key (case-insensitive)
        factory = self.factories.get(member_type.lower())

        if not factory:
            return False, {"message": "Invalid member type or factory not found", "member": None}

        member: Member = factory.create_member(name)

        self.library.members.append(member)
        self.notifier.attach(member)
        return True, {"message": f"Added {member_type} member: {name}", "member": f"{member_type} - {name}"}


class SearchBooksCommand(Command):

    def __init__(self, *args, **kwargs):
        """
        Initializes the SearchBooksCommand.

        Keyword Arguments:
        - library ('Library'): The library instance where the search will be performed.
        """
        self.library: 'Library' = kwargs.get('library')

    def execute(self, *args, **kwargs) -> Tuple[bool, Dict[str, Optional[str]]]:
        """
        Executes the search books command.

        Parameters:
        *args: Variable length argument list. (Not used in this implementation but provided for flexibility)
        **kwargs: Arbitrary keyword arguments. Expected keys include:
            - strategy (SearchStrategy): The search strategy to use (e.g., TitleSearchStrategy, AuthorSearchStrategy).
            - query (str): The query string to search for.

        Returns:
        Tuple[bool, Dict[str, Optional[str]]]: A tuple where:
            - The first element is a boolean indicating whether the search was successful (True) or not (False).
            - The second element is a dictionary containing additional information:
                - "available_books" (list): List of available books matching the search query.
                - "borrowed_books" (list): List of borrowed books matching the search query.
                - "message" (str): A message summarizing the search results.
        """
        strategy: SearchStrategy = kwargs.get('strategy')
        query: str = kwargs.get('query')

        if not strategy or not query:
            return False, {"message": "Invalid search parameters."}

        available_books, borrowed_books = strategy().search(self.library, query)
        result_text: str = ""

        if available_books or borrowed_books:
            if available_books:
                result_text += "Available books:\n"
                result_text += "\n".join([book.get_description() for book in available_books]) + "\n"
            if borrowed_books:
                result_text += "Borrowed books:\n"
                result_text += "\n".join([book.get_description() for book in borrowed_books])
        else:
            result_text = "No results found"

        return True, {
            "available_books": available_books,
            "borrowed_books": borrowed_books,
            "message": result_text
        }

class DisplayBorrowedBooksCommand(Command):

    def __init__(self, *args, **kwargs):
        """
        Initializes the DisplayBorrowedBooksCommand.

        Keyword Arguments:
        - library ('Library'): The library instance from which the borrowed books will be displayed.
        """
        self.library: 'Library' = kwargs.get('library')

    def execute(self, *args, **kwargs) -> Tuple[bool, Dict[str, Optional[str]]]:
        """
        Executes the display borrowed books command.

        Parameters:
        *args: Variable length argument list. (Not used in this implementation but provided for flexibility)
        **kwargs: Arbitrary keyword arguments. (Not used in this implementation but provided for flexibility)

        Returns:
        Tuple[bool, Dict[str, Optional[str]]]: A tuple where:
            - The first element is a boolean indicating whether the command was successful (True) or not (False).
            - The second element is a dictionary containing additional information:
                - "message" (str): A message summarizing the borrowed books.
                - "borrowed_books" (str): The descriptions of all borrowed books if any exist, or `None` if there are no borrowed books.
        """
        if not self.library.borrowed_books:
            return False, {"message": "No borrowed books found.", "borrowed_books": None}

        borrowed_books_descriptions = "\n".join([book.get_description() for book in self.library.borrowed_books])
        return True, {"message": "Borrowed Books:\n" + borrowed_books_descriptions, "borrowed_books": borrowed_books_descriptions}
    
class AddThirdPartyBooksCommand(Command):

    def __init__(self, *args, **kwargs):
        """
        Initializes the AddThirdPartyBooksCommand.

        Keyword Arguments:
        - library ('Library'): The library instance where the third-party books will be added.
        - notifier ('Notifier'): The notifier instance that will notify observers of the new third-party books.
        """
        self.library: 'Library' = kwargs.get('library')
        self.notifier: 'Notifier' = kwargs.get('notifier')  # Singleton instance of LibraryNotifier

    def execute(self, *args, **kwargs) -> Tuple[bool, Dict[str, Optional[str]]]:
        """
        Executes the add third-party books command.

        Parameters:
        *args: Variable length argument list. (Not used in this implementation but provided for flexibility)
        **kwargs: Arbitrary keyword arguments. (Not used in this implementation but provided for flexibility)

        Returns:
        Tuple[bool, Dict[str, Optional[str]]]: A tuple where:
            - The first element is a boolean indicating whether the command was successful (True) or not (False).
            - The second element is a dictionary containing additional information:
                - "message" (str): A message summarizing the result of the command.
                - "added_books" (Optional[str]): The descriptions of all books that were added, or `None` if no books were added.
        """
        third_party_api: ThirdPartyBookAPI = ThirdPartyBookAPI()
        third_party_books: list[dict] = third_party_api.get_books()
        if not third_party_books:
            return False, {"message": "No third-party books were found.", "added_books": None}

        added_books_descriptions = []

        for third_party_book in third_party_books:
            adapted_book: BookAdapter = BookAdapter(third_party_book)
            self.library.books.append(adapted_book)
            self.notifier.notify(book=adapted_book)
            added_books_descriptions.append(adapted_book.get_description())

        added_books_message = "\n".join(added_books_descriptions)
        return True, {
            "message": f"Third-party books added to the library:\n{added_books_message}",
            "added_books": added_books_message
        }
    
class DisplayAvailableBooksCommand(Command):

    def __init__(self, *args, **kwargs):
        """
        Initializes the DisplayAvailableBooksCommand.

        Keyword Arguments:
        - library ('Library'): The library instance from which the available books will be displayed.
        """
        self.library: 'Library' = kwargs.get('library')

    def execute(self, *args, **kwargs) -> Tuple[bool, Dict[str, Optional[str]]]:
        """
        Executes the display available books command.

        Parameters:
        *args: Variable length argument list. (Not used in this implementation but provided for flexibility)
        **kwargs: Arbitrary keyword arguments. (Not used in this implementation but provided for flexibility)

        Returns:
        Tuple[bool, Dict[str, Optional[str]]]: A tuple where:
            - The first element is a boolean indicating whether the command was successful (True) or not (False).
            - The second element is a dictionary containing additional information:
                - "message" (str): A message summarizing the available books.
                - "available_books" (Optional[str]): The descriptions of all available books if any exist, or `None` if there are no available books.
        """
        if not self.library.books:
            return False, {"message": "No available books found.", "available_books": None}

        available_books_descriptions = "\n".join([book.get_description() for book in self.library.books])
        return True, {"message": "Available Books:\n" + available_books_descriptions, "available_books": available_books_descriptions}
    
class DisplayUsersCommand(Command):

    def __init__(self, *args, **kwargs):
        """
        Initializes the DisplayUsersCommand.

        Keyword Arguments:
        - library ('Library'): The library instance from which the current users will be displayed.
        """
        self.library: 'Library' = kwargs.get('library')

    def execute(self, *args, **kwargs) -> Tuple[bool, Dict[str, Optional[str]]]:
        """
        Executes the display users command.

        Parameters:
        *args: Variable length argument list. (Not used in this implementation but provided for flexibility)
        **kwargs: Arbitrary keyword arguments. (Not used in this implementation but provided for flexibility)

        Returns:
        Tuple[bool, Dict[str, Optional[str]]]: A tuple where:
            - The first element is a boolean indicating whether the command was successful (True) or not (False).
            - The second element is a dictionary containing additional information:
                - "message" (str): A message summarizing the users.
                - "users" (Optional[str]): The descriptions of all current users if any exist, or `None` if there are no users.
        """
        if not self.library.members:
            return False, {"message": "No users found.", "users": None}

        users_descriptions = "\n".join([f"{type(member).__name__}: {member.name}" for member in self.library.members])
        return True, {"message": "Current Users:\n" + users_descriptions, "users": users_descriptions}


## Client / UI / Program Entry

In [26]:
# UI package - tkinter
import tkinter as tk
from tkinter import messagebox

# Config
SUPPORTED_BOOK_TYPES = ["EBook", "PrintBook"]
SEARCH_STRATEGIES = {
    "Search by Title": TitleSearchStrategy,
    "Search by Author": AuthorSearchStrategy
}
PREDEFINED_MEMBERS = [
    ("Alice", "Student"),
    ("Dr. Smith", "Faculty")
]
MEMBERTYPES = [
    ('Student', StudentMemberFactory),
    ('Faculty', FacultyMemberFactory),
]

# Client Class using Tkinter
class Client(tk.Tk):
    def __init__(self):
        super().__init__()

        ### Class Diagram Critical Info ###
        # 1. **Strategy Pattern**: `self.search_strategies` - used by `@Client.search_and_display_books`
        # 2. **Singleton Pattern**: Implemented in several classes:
        #    - `self.library` - `@Class.Library`
        #    - `self.notifier` - `@Class.LibraryNotifier`
        # 3. **Observer Pattern**: `self.notifier` - used by `@Class.AddBookCommand`, `@Class.AddMemberCommand`, and `@Class.AddThirdPartyBooksCommand`
        # 4. **Command Pattern**: Used extensively throughout the application:
        #    - `self.borrow_command` - `@Class.BorrowBookWithAccessoriesCommand`
        #    - `self.return_command` - `@Class.ReturnBookCommand`
        #    - `self.add_book_command` - `@Class.AddBookCommand`
        #    - `self.add_member_command` - `@Class.AddMemberCommand`
        #    - `self.search_books_command` - `@Class.SearchBooksCommand`
        #    - `self.display_borrowed_books_command` - `@Class.DisplayBorrowedBooksCommand`
        #    - `self.add_third_party_books_command` - `@Class.AddThirdPartyBooksCommand`
        #    - `self.display_available_books_command` - `@Class.DisplayAvailableBooksCommand`
        #    - `self.display_users_command` - `@Class.DisplayUsersCommand`
        # 5. **Adapter Pattern**: `@Class.BookAdapter` - used by `@Client.add_third_party_books_command`
        # 6. **Decorator Pattern**: Used in `@Client.prompt_for_accessories`, `@Class.BorrowBookWithAccessoriesCommand` with `@Class.BookAccessoryDecorator`
        #                              , `@Class.BookDecorator`
        # 7. **Factory Method Pattern**: `@Class.BookFactory` - used in `@Class.AddBookCommand`
        # 8. **Abstract Factory Pattern**: `@Class.StudentMemberFactory` and `@Class.FacultyMemberFactory` - used in `@Class.AddMemberCommand`
        ### Class Diagram Info End ###

        self.search_strategies: dict[str, SearchStrategy] = SEARCH_STRATEGIES   # 1. Strategy pattern object
        _library: Library = Library()                                           # 2. Singleton pattern object
        _notifier: LibraryNotifier = LibraryNotifier()                          # 3. Observer pattern object
        self.borrow_command: Command = BorrowBookWithAccessoriesCommand(library=_library)   # 4. Command pattern object
        self.return_command: Command = ReturnBookCommand(library=_library)                  # 4. Command pattern object
        self.add_book_command: Command = AddBookCommand(library=_library, notifier=_notifier)           # 4. Command pattern object

        member_factories = {member_type.lower(): factory() for member_type, factory in MEMBERTYPES}
        self.add_member_command: Command = AddMemberCommand(                                # 4. Command pattern object
            library=_library,
            notifier=_notifier,
            **member_factories  # Dynamically passing factories
        )

        self.search_books_command: Command = SearchBooksCommand(library=_library)                       # 4. Command pattern object
        self.display_borrowed_books_command: Command = DisplayBorrowedBooksCommand(library=_library)    # 4. Command pattern object
        self.add_third_party_books_command: Command = AddThirdPartyBooksCommand(library=_library, notifier=_notifier)  # 4. Command pattern object
        self.display_available_books_command: Command = DisplayAvailableBooksCommand(library=_library)  # 4. Command pattern object
        self.display_users_command: Command = DisplayUsersCommand(library=_library)                     # 4. Command pattern object

        # Title
        self.title("Library Management System")
        self.geometry("")
        self.create_widgets()
        # Configure grid layout to expand with window size
        self.grid_columnconfigure(1, weight=1)
        self.grid_columnconfigure(2, weight=1)
        self.grid_columnconfigure(3, weight=1)
        self.grid_columnconfigure(4, weight=1)
        self.grid_columnconfigure(5, weight=1)
        self.grid_rowconfigure(0, weight=1)
        self.grid_rowconfigure(1, weight=1)
        self.grid_rowconfigure(2, weight=1)

        # Initialize and register members for testing
        self.__init_members()

    def __init_members(self) -> None:
        """Initiate some members based on Configuration defined out of Client class scope for testing"""
        for name, member_type in PREDEFINED_MEMBERS:
            self.add_member_command.execute(name=name, member_type=member_type)

    def create_widgets(self) -> None:
        # Add book section
        tk.Label(self, text="Add Book").grid(row=0, column=0, padx=10, pady=10, sticky="e")

        # Dynamic selector for book type
        self.book_type_var: tk.StringVar = tk.StringVar(value=SUPPORTED_BOOK_TYPES[0])
        book_type_selector: tk.OptionMenu = tk.OptionMenu(self, self.book_type_var, *SUPPORTED_BOOK_TYPES)
        book_type_selector.grid(row=0, column=1, padx=10, pady=10, sticky="ew")

        self.title_var: tk.StringVar = tk.StringVar()
        self.author_var: tk.StringVar = tk.StringVar()

        tk.Label(self, text="Title:").grid(row=0, column=2, padx=10, pady=10, sticky="e")
        tk.Entry(self, textvariable=self.title_var).grid(row=0, column=3, padx=10, pady=10, sticky="ew")

        tk.Label(self, text="Author:").grid(row=0, column=4, padx=10, pady=10, sticky="e")
        tk.Entry(self, textvariable=self.author_var).grid(row=0, column=5, padx=10, pady=10, sticky="ew")

        tk.Button(self, text="Add", command=self.add_book).grid(row=0, column=6, padx=10, pady=10, sticky="ew")

        # Button to add third-party books
        tk.Button(self, text="Add Third-Party Books", command=self.add_third_party_books).grid(row=0, column=7, padx=10, pady=10, sticky="ew")

        # Search book section
        tk.Label(self, text="Search Book").grid(row=1, column=0, padx=10, pady=10, sticky="e")
        self.search_var: tk.StringVar = tk.StringVar()
        self.search_result_var: tk.StringVar = tk.StringVar()

        tk.Label(self, text="Query:").grid(row=1, column=1, padx=10, pady=10, sticky="e")
        tk.Entry(self, textvariable=self.search_var).grid(row=1, column=2, padx=10, pady=10, sticky="ew")

        # Dynamically create search buttons based on available strategies
        col: int = 3
        for strategy_name, strategy_class in self.search_strategies.items():
            tk.Button(self, text=strategy_name, command=lambda s=strategy_class: self.search_and_display_books(s)).grid(row=1, column=col, padx=10, pady=10, sticky="ew")
            col += 1

        tk.Label(self, textvariable=self.search_result_var).grid(row=1, column=col, padx=10, pady=10, sticky="ew")

        # Borrow and Return book section
        tk.Label(self, text="Borrow/Return Book").grid(row=2, column=0, padx=10, pady=10, sticky="e")
        self.borrow_return_var: tk.StringVar = tk.StringVar()

        tk.Label(self, text="Book Title:").grid(row=2, column=1, padx=10, pady=10, sticky="e")
        tk.Entry(self, textvariable=self.borrow_return_var).grid(row=2, column=2, padx=10, pady=10, sticky="ew")
        tk.Button(self, text="Borrow", command=self.borrow_book).grid(row=2, column=3, padx=10, pady=10, sticky="ew")
        tk.Button(self, text="Return", command=self.return_book).grid(row=2, column=4, padx=10, pady=10, sticky="ew")

        # Add member section
        tk.Label(self, text="Add Member").grid(row=3, column=0, padx=10, pady=10, sticky="e")

        self.member_name_var: tk.StringVar = tk.StringVar()
        self.member_type_var: tk.StringVar = tk.StringVar(value=MEMBERTYPES[0][0])  # Default to the first type in MEMBERTYPES

        tk.Label(self, text="Name:").grid(row=3, column=1, padx=10, pady=10, sticky="e")
        tk.Entry(self, textvariable=self.member_name_var).grid(row=3, column=2, padx=10, pady=10, sticky="ew")

        # Create an OptionMenu for member types based on MEMBERTYPES
        member_type_names = [member_type[0] for member_type in MEMBERTYPES]
        member_type_selector: tk.OptionMenu = tk.OptionMenu(self, self.member_type_var, *member_type_names)
        member_type_selector.grid(row=3, column=3, padx=10, pady=10, sticky="ew")

        tk.Button(self, text="Add Member", command=self.add_member).grid(row=3, column=4, padx=10, pady=10, sticky="ew")


        # Button to display current users
        tk.Button(self, text="Show Current Users", command=self.display_users).grid(row=3, column=7, padx=10, pady=10, sticky="ew")

        # Info Display Button
        tk.Button(self, text="Show Available Books", command=self.display_available_books).grid(row=2, column=6, padx=10, pady=10, sticky="ew")
        tk.Button(self, text="Show Borrowed Books", command=self.display_borrowed_books).grid(row=2, column=5, padx=10, pady=10, sticky="ew")

    def add_member(self) -> None:
        name: str = self.member_name_var.get().strip()
        member_type: str = self.member_type_var.get()

        if not name:
            messagebox.showerror("Error", "Name cannot be empty or contain only whitespace.")
            return

        success: bool
        data: dict
        success, data = self.add_member_command.execute(name=name, member_type=member_type)

        if success:
            messagebox.showinfo("Success", data["message"])
        else:
            messagebox.showerror("Error", data["message"])

    def add_book(self) -> None:
        book_type: str = self.book_type_var.get()
        title: str = self.title_var.get().strip()
        author: str = self.author_var.get()

        if not title:
            messagebox.showerror("Error", "Title cannot be empty or contain only whitespace.")
            return

        success: bool
        data: dict
        success, data = self.add_book_command.execute(book_type=book_type, title=title, author=author)

        if success and data["book"]:
            messagebox.showinfo("Success", f"{data['message']}")
        else:
            messagebox.showerror("Error", f"{data['message']}")

    def search_and_display_books(self, strategy_class: SearchStrategy) -> None:
        query: str = self.search_var.get().strip()

        if not query:
            messagebox.showerror("Error", "Keyword cannot be empty or contain only whitespace.")
            return

        success: bool
        data: dict
        success, data = self.search_books_command.execute(strategy=strategy_class, query=query)

        if success:
            self.search_result_var.set(data["message"])
            messagebox.showinfo("Search Results", data["message"])
        else:
            messagebox.showerror("Error", data["message"])
            

    def borrow_book(self) -> None:
        target_title: str = self.borrow_return_var.get().strip()

        if not target_title:
            messagebox.showerror("Error", "Title cannot be empty or contain only whitespace.")
            return

        accessories: list[str] = self.__prompt_for_accessories()
        
        success: bool
        data: dict
        success, data = self.borrow_command.execute(title=target_title, accessories=accessories)

        messagebox.showinfo("Result", data['message'])

    def __prompt_for_accessories(self) -> list[str]:
        """
        Display a window to ask user to add asseccories when borrowing a book.

        Returns:
        - A list of accessory names
        """

        accessory_window: tk.Toplevel = tk.Toplevel(self)
        accessory_window.title("Add Accessories")

        tk.Label(accessory_window, text="Enter accessory:").grid(row=0, column=0, padx=10, pady=10)
        accessory_var: tk.StringVar = tk.StringVar()
        tk.Entry(accessory_window, textvariable=accessory_var).grid(row=0, column=1, padx=10, pady=10)

        accessories: list[str] = []

        def add_accessory() -> None:
            accessory: str = accessory_var.get()
            if accessory:
                accessories.append(accessory)
                accessory_var.set("")  # Clear the entry field

        def done() -> None:
            accessory_window.destroy()

        tk.Button(accessory_window, text="Add Accessory", command=add_accessory).grid(row=1, column=0, padx=10, pady=10)
        tk.Button(accessory_window, text="Done", command=done).grid(row=1, column=1, padx=10, pady=10)

        accessory_window.transient(self)
        accessory_window.grab_set()
        self.wait_window(accessory_window)
        return accessories


    def return_book(self) -> None:
        target_title: str = self.borrow_return_var.get().strip()

        if not target_title:
            messagebox.showerror("Error", "Title cannot be empty or contain only whitespace.")
            return

        success: bool
        data: dict
        success, data = self.return_command.execute(title=target_title)

        messagebox.showinfo("Result", f"{data['message']}")

    def display_available_books(self) -> None:
        success: bool
        data: dict
        success, data = self.display_available_books_command.execute()

        if success:
            messagebox.showinfo("Available Books", data["message"])
        else:
            messagebox.showinfo("No Available Books", data["message"])

    def display_borrowed_books(self) -> None:
        success: bool
        data: dict
        success, data = self.display_borrowed_books_command.execute()

        if success:
            messagebox.showinfo("Borrowed Books", data["message"])
        else:
            messagebox.showinfo("No Borrowed Books", data["message"])

    def add_third_party_books(self) -> None:
        success: bool
        data: dict
        success, data = self.add_third_party_books_command.execute()

        if success:
            messagebox.showinfo("Success", data["message"])
        else:
            messagebox.showerror("Error", data["message"])

    def display_users(self) -> None:
        success: bool
        data: dict
        success, data = self.display_users_command.execute()

        if success:
            messagebox.showinfo("Current Users", data["message"])
        else:
            messagebox.showinfo("No Users", data["message"])

In [27]:
client = Client()
client.mainloop()